# Задание 4. Декораторы на практике

**Выполнил:** WhyNot-png (kohen). Группа не указана.

Копия учебного ноутбука «Лабораторная работа 3. Декораторы на практике».
[Исходный материал](https://drive.google.com/file/d/1sgFLmxu1B67WYfdvcCKRRCELGoye4TvV/view).
Автор исходной заготовки: Жуков Николай.

**Выбрано упражнение 2, выполнены пункты 2.1–2.4.** Написаны настройка логгера,
декораторы, запуски тестовых функций и ответы на вопросы. Проверочные ячейки 2.1–2.4
сохранены без изменений. Функция `find_extrema` не изменена.

Упражнение 1 не выбрано: его исходные заготовки сохранены как текст, чтобы они
не мешали команде «Перезапустить сеанс и выполнить все». Бонусы не заявляются.
Теоретические примеры сохранены. Все зависимости — стандартная библиотека Python.
Файлы `extrema.log` и `extrema_info.log` создаются при выполнении и выведены в ноутбуке.

---
# Часть A. Теория для самостоятельного изучения

## A1. Positional-only параметры (`/`)

Параметры **до** символа `/` можно передать **только позиционно** (Python 3.8+). Зачем это нужно:
- имя параметра не становится частью интерфейса: автор функции может его переименовать, и чужой код не сломается;
- так устроены многие встроенные функции: `len(obj=[1, 2])` не работает.

In [1]:
def power(base, exp, /, *, mod=None):
    """base и exp — только позиционно, mod — только по ключу."""
    result = base ** exp
    return result % mod if mod is not None else result

print(power(2, 10))
print(power(2, 10, mod=1000))

try:
    power(base=2, exp=10)
except TypeError as e:
    print("TypeError:", e)

try:
    len(obj=[1, 2])
except TypeError as e:
    print("TypeError:", e)

1024
24
TypeError: power() got some positional-only arguments passed as keyword arguments: 'base, exp'
TypeError: len() takes no keyword arguments


## A2. Ловушка: изменяемое значение по умолчанию

Значения по умолчанию вычисляются **один раз — при выполнении `def`**, а не при каждом вызове. Если значение по умолчанию изменяемое (список, словарь), все вызовы работают с **одним и тем же** объектом.

Сравните с декораторами: код декоратора вне обёртки тоже выполняется один раз — при определении функции.

In [2]:
def append_to(item, target=[]):
    target.append(item)
    return target

print(append_to(1))
print(append_to(2))            # ожидали [2], а получили [1, 2]
print(append_to.__defaults__)  # «общий» список хранится в самой функции

[1]
[1, 2]
([1, 2],)


Правильный приём — использовать `None` как маркер «значение не передано» и создавать новый объект внутри функции:

In [3]:
def append_to(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target

print(append_to(1))
print(append_to(2))

[1]
[2]


## A3. `global` и затенение встроенных имён

### `global`

`global name` позволяет **присвоить** значение глобальной переменной изнутри функции. Пользуйтесь этим редко: функция, которая меняет глобальное состояние, зависит от всего окружения, её сложно тестировать и повторно использовать. Обычно лучше вернуть значение из функции или хранить состояние в замыкании или объекте.

In [4]:
total = 0

def add_to_total(x):
    global total
    total += x

add_to_total(5)
add_to_total(7)
print(total)

12


### Затенение встроенных имён

Встроенные имена (`sum`, `list`, `str`, `id`, `max`…) находятся в самой внешней области **B**. Если создать переменную с таким же именем в области **G** или **L**, она «затенит» встроенную функцию, и та станет недоступна по этому имени.

In [5]:
sum = 10                       # затенили встроенную sum в глобальной области
try:
    print(sum([1, 2, 3]))
except TypeError as e:
    print("TypeError:", e)

del sum                        # удалили глобальное имя → снова видна встроенная
print(sum([1, 2, 3]))

import builtins
print(builtins.sum is sum)     # встроенные имена живут в модуле builtins

TypeError: 'int' object is not callable
6
True


> Не называйте переменные `list`, `dict`, `str`, `id`, `input`, `sum`, `max`, `min`, `type`. Ошибка проявится не там, где вы создали переменную, а там, где кто-то попробует вызвать встроенную функцию.

## A4. Позднее связывание в замыканиях

Замыкание хранит **ссылку на переменную**, а не её значение в момент создания функции. Значение читается **в момент вызова**.

In [6]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])     # ожидали [0, 1, 2]

# Все три lambda ссылаются на одну переменную i,
# а к моменту вызова цикл закончился и i == 2.

[2, 2, 2]


Два способа «зафиксировать» значение:
1. значение по умолчанию (вычисляется при создании функции, см. A2);
2. фабрика функций: каждый вызов создаёт новую область видимости со своей переменной.

In [7]:
funcs = [lambda i=i: i for i in range(3)]          # способ 1
print([f() for f in funcs])

def make(i):                                       # способ 2
    return lambda: i

funcs = [make(i) for i in range(3)]
print([f() for f in funcs])

[0, 1, 2]
[0, 1, 2]


## A5. Три способа хранить состояние между вызовами

Задача: функция `avg(x)` принимает очередное число и возвращает среднее всех чисел, переданных до сих пор.

### Способ 1. Замыкание + `nonlocal`

In [8]:
def make_averager():
    total = 0
    count = 0

    def averager(x):
        nonlocal total, count
        total += x
        count += 1
        return total / count

    return averager

avg = make_averager()
print(avg(10), avg(20), avg(60))

10.0 15.0 30.0


Состояние спрятано: снаружи до `total` и `count` не добраться (разве что через `__closure__`).

### Способ 2. Атрибут функции

Функция — объект, у неё есть `__dict__`, и ей можно добавить атрибуты. Состояние видно и доступно снаружи.

In [9]:
def avg(x):
    avg.total += x
    avg.count += 1
    return avg.total / avg.count

avg.total = 0
avg.count = 0

print(avg(10), avg(20), avg(60))
print(avg.count)

10.0 15.0 30.0
3


### Способ 3. Класс с методом `__call__`

Экземпляр класса, в котором определён метод `__call__`, можно вызывать как функцию. Подходит, когда состояния много или нужны дополнительные методы (например, `reset`).

In [10]:
class Averager:
    def __init__(self):
        self.total = 0
        self.count = 0

    def __call__(self, x):
        self.total += x
        self.count += 1
        return self.total / self.count

    def reset(self):
        self.total = self.count = 0

avg = Averager()
print(avg(10), avg(20), avg(60))
avg.reset()
print(avg(5))

10.0 15.0 30.0
5.0


Все три способа применимы и к декораторам. Пример **декоратора-класса**: сам класс принимает функцию в `__init__`, а экземпляр заменяет её и вызывается через `__call__`. Метаданные копируются через `functools.update_wrapper(self, func)`.

In [11]:
import functools

class Remember:
    """Декоратор-класс: запоминает последний результат функции."""
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.last_result = None

    def __call__(self, *args, **kwargs):
        self.last_result = self.func(*args, **kwargs)
        return self.last_result

@Remember
def hypot(a, b):
    """Длина гипотенузы."""
    return (a * a + b * b) ** 0.5

hypot(3, 4)
print(hypot.last_result, hypot.__name__, hypot.__doc__)
print(type(hypot))

5.0 hypot Длина гипотенузы.
<class '__main__.Remember'>


| Способ | Плюсы | Минусы |
|--------|-------|--------|
| Замыкание | состояние скрыто, код компактный | нужен `nonlocal`; состояние не видно снаружи |
| Атрибут функции | просто, состояние видно снаружи | состояние можно случайно испортить снаружи |
| Класс с `__call__` | много состояния, дополнительные методы | больше кода; результат — не функция, а объект класса |

## A6. Декоратор `once`

Декоратор `once` позволяет выполнить функцию только при первом вызове. Это бывает полезно для инициализации настроек, подключения к ресурсу и т. п. Состояние «уже вызывалась» хранится в атрибуте обёртки.

In [12]:
import functools

def once(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if not wrapper.called:
            wrapper.called = True
            func(*args, **kwargs)
    wrapper.called = False
    return wrapper

@once
def initialize_settings():
    print("Настройки инициализированы.")
    return {"theme": "dark"}

print(initialize_settings())    # функция выполнилась, но результат потерян!
print(initialize_settings())    # второй вызов ничего не делает

Настройки инициализированы.
None
None


> У этой реализации есть недостаток: обёртка не возвращает результат. Исправление декоратора `once` — задание **домашней работы**.

## A7. Декоратор `deprecated`

Помечает функцию как устаревшую. Правильно выдавать предупреждение **при вызове** функции, а не при её определении (иначе предупреждение увидит тот, кто импортирует модуль, а не тот, кто вызывает устаревшую функцию).

`warnings.warn(..., stacklevel=2)` указывает в сообщении строку, **откуда функцию вызвали**, а не строку внутри обёртки.

In [13]:
import functools
import warnings

def deprecated(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        warnings.warn(
            f"Функция {func.__name__} устарела и будет удалена.",
            category=DeprecationWarning,
            stacklevel=2,
        )
        return func(*args, **kwargs)
    return wrapper

@deprecated
def old_sum(a, b):
    return a + b

warnings.simplefilter("always", DeprecationWarning)   # показывать каждый раз
print(old_sum(1, 2))

3


/tmp/ipykernel_6579/694249063.py:20: DeprecationWarning: Функция old_sum устарела и будет удалена.
  print(old_sum(1, 2))


По умолчанию Python показывает `DeprecationWarning` только для кода в главном модуле и только **один раз** для каждого места вызова. Строка `warnings.simplefilter("always", ...)` включает показ при каждом вызове.

## A8. Контрактное программирование: `pre` и `post`

**Контракт** — формальное описание того, что функция ожидает на входе (**предусловие**) и что гарантирует на выходе (**постусловие**). Декоратор позволяет вынести проверки из тела функции.

In [14]:
import functools
import math

def pre(cond, message):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            assert cond(*args, **kwargs), message
            return func(*args, **kwargs)
        return wrapper
    return decorator

def post(cond, message):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            assert cond(result), message
            return result
        return wrapper
    return decorator

@pre(lambda x: x > 0, "аргумент должен быть положительным")
def checked_log(x):
    return math.log(x)

@post(lambda r: not math.isnan(r), "результат — не число (NaN)")
def something_useful():
    return float("nan")

for call in (lambda: checked_log(-42), something_useful):
    try:
        call()
    except AssertionError as e:
        print("AssertionError:", e)

AssertionError: аргумент должен быть положительным
AssertionError: результат — не число (NaN)


**Важно:** инструкции `assert` удаляются, если запустить Python с оптимизацией (`python -O`). Тогда контракты молча перестанут проверяться. Для проверок, которые должны работать всегда, выбрасывайте исключение явно:

In [15]:
import functools
import math

class ContractError(Exception):
    """Нарушение контракта функции."""

def pre(cond, message):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if not cond(*args, **kwargs):
                raise ContractError(f"{func.__name__}: {message}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@pre(lambda x: x > 0, "аргумент должен быть положительным")
def checked_log(x):
    return math.log(x)

try:
    checked_log(-1)
except ContractError as e:
    print("ContractError:", e)

ContractError: checked_log: аргумент должен быть положительным


## A9. Для любознательных: «декоратор для декораторов»

Писать три уровня вложенности для каждого декоратора с аргументами утомительно. Можно написать декоратор `with_arguments`, который превращает обычную функцию `deco(func, *args, **kwargs)` в декоратор с аргументами.

In [16]:
import functools
import sys

def with_arguments(deco):
    @functools.wraps(deco)
    def factory(*dargs, **dkwargs):          # принимает аргументы декоратора
        def decorator(func):                 # принимает функцию
            result = deco(func, *dargs, **dkwargs)
            functools.update_wrapper(result, func)
            return result
        return decorator
    return factory

@with_arguments
def trace(func, handle):                     # всего два уровня вместо трёх
    def wrapper(*args, **kwargs):
        print(func.__name__, args, kwargs, file=handle)
        return func(*args, **kwargs)
    return wrapper

@trace(sys.stdout)
def increment(x):
    """Инкремент."""
    return x + 1

print(increment(41), increment.__name__, increment.__doc__)

increment (41,) {}
42 increment Инкремент.


Разберите по шагам: что такое `trace` после применения `@with_arguments`? Что возвращает `trace(sys.stdout)`?

## A10. Модуль `functools`

### `partial` — частичное применение

`functools.partial(func, *args, **kwargs)` создаёт новый вызываемый объект, у которого часть аргументов уже зафиксирована.

In [17]:
from functools import partial

basetwo = partial(int, base=2)
print(basetwo("10010"))

def add(x, *ys, k=None):
    return x, ys, k

p_add = partial(add, 2, 100, k="fixed")
print(p_add("z", "yy"))                  # позиционные дописываются в конец
print(p_add.func, p_add.args, p_add.keywords)

18
(2, (100, 'z', 'yy'), 'fixed')
<function add at 0x7fb4f023ba60> (2, 100) {'k': 'fixed'}


`partial` удобно использовать в декораторах с необязательными аргументами вместо `lambda`:

```python
def trace(func=None, *, handle=sys.stdout):
    if func is None:
        return functools.partial(trace, handle=handle)
    ...
```

### `singledispatch` — обобщённые функции

**Обобщённая функция** выбирает реализацию в зависимости от **типа** аргумента. Встроенные примеры: `len`, `str`, `hash` работают по-разному для разных типов. Декоратор `functools.singledispatch` позволяет написать такую функцию без цепочки `if isinstance(...)`. Выбор делается по типу **первого** аргумента.

Пример: функция `pack` сериализует объект в компактное байтовое представление.

In [18]:
import functools

@functools.singledispatch
def pack(obj):
    raise TypeError(f"Неподдерживаемый тип: {type(obj).__name__}")

@pack.register
def _(obj: int):                         # тип можно указать аннотацией
    return b"I" + hex(obj).encode("ascii")

@pack.register(list)                     # или аргументом register
@pack.register(tuple)                    # одну реализацию — для двух типов
def _(obj):
    return b"L" + b",".join(map(pack, obj))

print(pack([1, 2, 255]))
print(pack((10, [11, 12])))
try:
    pack("строка")
except TypeError as e:
    print("TypeError:", e)
print(pack.registry.keys())

b'LI0x1,I0x2,I0xff'
b'LI0xa,LI0xb,I0xc'
TypeError: Неподдерживаемый тип: str
dict_keys([<class 'object'>, <class 'int'>, <class 'tuple'>, <class 'list'>])


### `reduce` — свёртка

`functools.reduce(op, iterable, initial)` последовательно применяет бинарную операцию слева направо:

```python
reduce(op, [x0, x1, x2], initial)  ==  op(op(op(initial, x0), x1), x2)
reduce(op, [x0, x1, x2])           ==  op(op(x0, x1), x2)
```

In [19]:
import functools
import operator

print(functools.reduce(operator.mul, [1, 2, 3, 4, 5]))                  # 120
print(functools.reduce(lambda acc, d: 10 * acc + int(d), "2026", 0))    # строка → число
print(functools.reduce(lambda a, b: a + b, [[1, 2, 7], [5, 6], [0]], []))

try:
    functools.reduce(operator.add, [])
except TypeError as e:
    print("TypeError:", e)     # без initial пустую последовательность свернуть нельзя

120
2026
[1, 2, 7, 5, 6, 0]
TypeError: reduce() of empty iterable with no initial value


Для частых случаев есть готовые функции: `sum`, `math.prod`, `max`, `min`, `"".join`. Их стоит предпочитать `reduce`: они быстрее и читаются проще.

## A11. Декораторы в реальном коде

### Встроенные декораторы для классов и методов

In [20]:
from dataclasses import dataclass

class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property                          # метод читается как атрибут
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @staticmethod                      # не получает ни self, ни cls
    def is_valid(celsius):
        return celsius >= -273.15

    @classmethod                       # получает класс: альтернативный конструктор
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)

t = Temperature.from_fahrenheit(212)
print(t.celsius, t.fahrenheit, Temperature.is_valid(-300))

@dataclass                             # декоратор КЛАССА: генерирует __init__, __repr__, __eq__
class Point:
    x: float
    y: float = 0.0

print(Point(1, 2), Point(1, 2) == Point(1, 2))

100.0 212.0 False
Point(x=1, y=2) True


### Декораторы в веб-фреймворках

В веб-фреймворках декораторы регистрируют функции-обработчики HTTP-запросов (ячейка не выполняется — это иллюстрация):

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/students/{student_id}")      # при импорте модуля функция попадает в таблицу маршрутов
async def get_student(student_id: int):
    return {"id": student_id}
```

`app.get("/students/{student_id}")` — фабрика декораторов, как `repeat(3)` на лекции. Декоратор запоминает функцию в таблице маршрутов и возвращает её же. Когда приходит запрос на этот адрес, фреймворк находит функцию в таблице и вызывает её.

### Вопросы для самопроверки по части A

1. Почему `def f(x, cache={})` можно использовать как «кэш», и почему так делать не стоит?
2. Что выведет `[f() for f in [lambda: i * 2 for i in range(3)]]`? Как исправить?
3. Когда декоратор-класс удобнее декоратора-функции?
4. Почему `deprecated` должен предупреждать при вызове, а не при определении функции?
5. Чем опасна проверка контракта через `assert`?

---
# Часть B. Практика

Выполните ячейку с импортами — все упражнения используют эти модули.

In [21]:
import functools
import logging
import math
import reprlib
import sys
import time

## Упражнение 1. Сколько стоит точность: таймер для численного интегрирования

**Цель:** написать декоратор без аргументов и с необязательными аргументами, сохранить состояние в атрибуте обёртки, применить несколько декораторов и измерить накладные расходы декоратора.

### Исходный код

Функция `integrate` вычисляет определённый интеграл $\int_a^b f(x)\,dx$ **методом левых прямоугольников**: отрезок делится на `n_iter` частей, и площадь под графиком приближается суммой площадей прямоугольников высотой $f(x_i)$ в левой точке каждой части:

$$\int_a^b f(x)\,dx \approx \sum_{i=0}^{n-1} f(a + i h)\cdot h, \qquad h = \frac{b-a}{n}.$$

Для проверки точности используем $\int_0^1 e^x\,dx = e - 1$: точное значение известно, поэтому можно посчитать погрешность.

**Изменять функцию `integrate` нельзя** — всё дополнительное поведение добавляется только декораторами.

```python
def integrate(f, a, b, *, n_iter=1000):
    """Интеграл f на [a, b] методом левых прямоугольников."""
    acc = 0
    step = (b - a) / n_iter
    for i in range(n_iter):
        acc += f(a + i * step) * step
    return acc

EXACT = math.e - 1
value = integrate(math.exp, 0, 1, n_iter=10**4)
print(value, "погрешность:", abs(value - EXACT))
```

### 1.1. Декоратор `timer` (1 балл)

Напишите декоратор `timer` **без аргументов**, который:
- измеряет время выполнения функции с помощью `time.perf_counter()`;
- выводит строку вида `integrate: 0.0123 с`;
- возвращает результат исходной функции без изменений;
- сохраняет имя и документацию функции (`functools.wraps`).

**Подсказка:** используйте каноническую обёртку из лекции (раздел 5.3). Замер — это разность двух вызовов `time.perf_counter()` до и после вызова функции.

```python
def timer(func):
    # TODO: реализуйте декоратор
    ...
```

```python
# ПРОВЕРКА 1.1 — не изменяйте эту ячейку

@timer                                   # применение через синтаксический сахар
def slow_add(a, b):
    """Медленное сложение."""
    time.sleep(0.05)
    return a + b

assert slow_add(2, 3) == 5, "обёртка должна возвращать результат функции"
assert slow_add.__name__ == "slow_add", "используйте functools.wraps"
assert slow_add.__doc__ == "Медленное сложение."

timed_integrate = timer(integrate)       # то же, что @timer над def integrate
assert timed_integrate(math.exp, 0, 1, n_iter=1000) == integrate(math.exp, 0, 1, n_iter=1000)
print("1.1 — все проверки пройдены")
```

### 1.2. `timer` с необязательными аргументами (2 балла)

Однократный замер неточен: на время влияют другие процессы. Перепишите `timer` так, чтобы:

1. Декоратор работал **в обеих формах**: `@timer` и `@timer(repeat=5, verbose=False)`.
2. Параметры — **только ключевые** (keyword-only):
   - `repeat=1` — сколько раз выполнить функцию при одном вызове обёртки;
   - `verbose=True` — печатать ли строку с временем.
3. Обёртка выполняет функцию `repeat` раз и запоминает **минимальное** время.
4. Минимальное время сохраняется в **атрибут обёртки** `last_time` (в секундах). До первого вызова `last_time` равен `None`.
5. При `verbose=True` печатается строка вида `integrate: 0.0123 с (лучшее из 5)`.
6. Возвращается результат функции (последнего из `repeat` вызовов).

**Подсказки:**
- шаблон декоратора с необязательными аргументами — в лекции, раздел 7.2;
- хранение состояния в атрибуте обёртки — пример `profiled` из лекции, раздел 9.2.

```python
def timer(func=None, *, repeat=1, verbose=True):
    # TODO: если func is None — вернуть декоратор (лекция, раздел 7.2)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # TODO: выполнить func repeat раз, найти минимальное время,
        #       сохранить его в wrapper.last_time, при verbose — напечатать,
        #       вернуть результат
        ...

    # TODO: начальное значение атрибута last_time
    return wrapper
```

```python
# ПРОВЕРКА 1.2 — не изменяйте эту ячейку

calls = 0

def work(x):
    """Удваивает x."""
    global calls
    calls += 1
    return x * 2

t1 = timer(work)                                    # форма @timer
assert t1.last_time is None, "до первого вызова last_time должен быть None"
assert t1(5) == 10 and calls == 1
assert t1.last_time is not None and t1.last_time >= 0

calls = 0
t3 = timer(repeat=3, verbose=False)(work)           # форма @timer(...)
assert t3(7) == 14
assert calls == 3, f"функция должна выполниться repeat=3 раза, а выполнилась {calls}"
assert t3.__name__ == "work" and t3.__doc__ == "Удваивает x."

try:
    timer(work, 3)
except TypeError:
    pass
else:
    raise AssertionError("repeat и verbose должны быть keyword-only")

print("1.2 — все проверки пройдены")
```

### 1.3. Эксперимент: время и погрешность (1 балл)

Для `n_iter` из списка `[10**3, 10**4, 10**5, 10**6]`:
1. Вычислите интеграл $\int_0^1 e^x\,dx$ функцией `integrate`, обёрнутой в `timer(repeat=3, verbose=False)`.
2. Для каждого `n_iter` выведите строку таблицы: `n_iter`, время (из `last_time`), абсолютную погрешность `abs(value - EXACT)`.

Пример формата вывода:
```
   n_iter    время, с    погрешность
     1000      0.0002       8.59e-04
```

**❓ Вопросы** (ответьте в markdown-ячейке ниже):
1. Во сколько раз растёт время при увеличении `n_iter` в 10 раз? Почему?
2. Во сколько раз уменьшается погрешность? Какой вывод можно сделать о «цене» каждого дополнительного верного знака?

```python
timed_integrate = timer(integrate, repeat=3, verbose=False)

print(f"{'n_iter':>9} {'время, с':>11} {'погрешность':>14}")
for n in [10**3, 10**4, 10**5, 10**6]:
    # TODO: вычислить интеграл, взять время из timed_integrate.last_time,
    #       посчитать погрешность и вывести строку таблицы
    ...
```

**Упражнение 1.3 не выбрано.**

### 1.4. Сколько раз вызвана подынтегральная функция? (1 балл)

Напишите декоратор `count_calls` **без аргументов**, который:
- считает вызовы функции в атрибуте обёртки `calls` (начальное значение `0`);
- возвращает результат исходной функции.

Затем:
1. Задекорируйте **подынтегральную** функцию `counted_exp` (заготовка ниже) и убедитесь, что `integrate` вызывает её ровно `n_iter` раз.
2. Сравните время `integrate(math.exp, ...)` и `integrate(counted_exp, ...)` при `n_iter=10**6` (используйте `timer` из 1.2) и выведите, **во сколько раз** медленнее работает вариант с декоратором.

**❓ Вопросы:**
1. Почему декоратор `count_calls` применяется к `f`, а не к `integrate`?
2. Откуда берётся замедление? Стоит ли навешивать декораторы на функции, которые вызываются миллионы раз?

```python
def count_calls(func):
    # TODO: реализуйте декоратор
    ...


@count_calls
def counted_exp(x):
    return math.exp(x)


# TODO: сравнить время integrate(math.exp, 0, 1, n_iter=10**6)
#       и integrate(counted_exp, 0, 1, n_iter=10**6), вывести отношение
```

```python
# ПРОВЕРКА 1.4 — не изменяйте эту ячейку

counted_exp.calls = 0
integrate(counted_exp, 0, 1, n_iter=1234)
assert counted_exp.calls == 1234, f"ожидалось 1234 вызова, получено {counted_exp.calls}"
assert counted_exp.__name__ == "counted_exp"
assert counted_exp(0) == 1.0
print("1.4 — все проверки пройдены")
```

**Упражнение 1.4 не выбрано.**

### Бонус 1. Метод средних прямоугольников (+1 балл, необязательно)

В методе **средних** прямоугольников высота берётся в **середине** каждой части: $f(a + (i + \tfrac12) h)$.

1. Напишите функцию `integrate_mid` с той же сигнатурой, что у `integrate`, и сразу задекорируйте её `@timer(repeat=3, verbose=False)`.
2. Повторите таблицу из 1.3 для обоих методов.
3. **❓** Как быстро убывает погрешность каждого метода? Во сколько раз можно уменьшить `n_iter` методом средних прямоугольников при той же точности?

Обратите внимание: так как `integrate_mid` задекорирована через `@`, её исходная версия без таймера доступна как `integrate_mid.__wrapped__`.

```python
# Бонус 1 (необязательно)
```

## Упражнение 2. Журнал поиска экстремумов: декораторы и `logging`

**Цель:** написать декоратор, который записывает в журнал (лог) параметры и результаты вызова функции, освоить уровни логирования и научиться комбинировать декораторы.

### Модуль `logging`: что нужно знать

**Почему не `print`?** Сообщения журнала имеют **уровень важности**, их можно фильтровать, направлять одновременно в файл и на экран, снабжать временем и именем источника — и всё это без изменения кода, который пишет сообщения.

**Уровни логирования** (по возрастанию важности):

| Уровень | Число | Когда использовать | В этом упражнении |
|---------|-------|--------------------|-------------------|
| `DEBUG` | 10 | подробности для отладки | аргументы вызова, время работы |
| `INFO` | 20 | штатные события | результат вызова |
| `WARNING` | 30 | что-то подозрительное, но работа продолжается | экстремум на границе отрезка |
| `ERROR` | 40 | ошибка: операция не выполнена | исключение в функции |
| `CRITICAL` | 50 | программа не может продолжать работу | — |

**Три участника:**
- **Logger** (логгер) — объект, в который код пишет сообщения: `logger.info("...")`. Получается через `logging.getLogger("имя")`: один и тот же объект для одного имени.
- **Handler** (обработчик) — куда отправлять сообщения: `FileHandler` (файл), `StreamHandler` (поток, например экран). У логгера может быть несколько обработчиков.
- **Formatter** (форматировщик) — как выглядит строка: время, уровень, имя логгера, текст.

**Фильтрация по уровням происходит дважды:** сообщение сначала сравнивается с уровнем **логгера**, затем — с уровнем **каждого обработчика**. Сообщение попадёт в обработчик, только если его уровень не ниже обоих порогов.

```
logger.debug("...")  ──▶  Logger (level=DEBUG) ──┬──▶ FileHandler   (level=DEBUG)   ──▶ extrema.log  (всё)
                                                  └──▶ StreamHandler (level=WARNING) ──▶ экран (только WARNING и выше)
```

**Форматирование сообщений.** В `logging` принято передавать шаблон и аргументы отдельно: `logger.debug("x = %s", x)`, а не `logger.debug(f"x = {x}")`. Строка собирается, только если сообщение действительно будет выведено, — это экономит время на отключённых уровнях.

**Ловушка Colab.** В Colab корневой логгер обычно уже настроен средой, поэтому `logging.basicConfig(...)` **молча ничего не делает**. Проверить можно так: `print(logging.getLogger().handlers)`. Решения: `logging.basicConfig(..., force=True)` или, лучше, собственный именованный логгер с явно добавленными обработчиками — так и сделаем.

In [22]:
print("Обработчики корневого логгера:", logging.getLogger().handlers)

Обработчики корневого логгера: []


### Исследуемая функция

Функция `find_extrema(f, a, b, *, n=1000)` ищет экстремумы функции `f` на отрезке `[a, b]` **перебором по сетке**: вычисляет `f` в `n + 1` равноотстоящих точках и сравнивает каждое значение с соседями.
- Точка — **локальный минимум**, если значение в ней меньше, чем в левой соседней, и не больше, чем в правой (для максимума — наоборот).
- **Глобальный** минимум и максимум ищутся среди всех точек, **включая концы отрезка**.

Функция возвращает словарь:
```python
{
    "local_min":  [(x, f(x)), ...],
    "local_max":  [(x, f(x)), ...],
    "global_min": (x, f(x)),
    "global_max": (x, f(x)),
    "interval":   (a, b),
}
```
При `a >= b` или `n < 2` выбрасывается `ValueError`.

**Изменять функцию `find_extrema` нельзя** — журналирование добавляется только декораторами.

In [23]:
def find_extrema(f, a, b, *, n=1000):
    """Поиск локальных и глобальных экстремумов f на [a, b] перебором по сетке."""
    if a >= b:
        raise ValueError(f"левая граница должна быть меньше правой: a={a}, b={b}")
    if n < 2:
        raise ValueError(f"число разбиений должно быть не меньше 2: n={n}")

    step = (b - a) / n
    xs = [a + i * step for i in range(n + 1)]
    ys = [f(x) for x in xs]

    local_min, local_max = [], []
    for i in range(1, n):
        if ys[i - 1] > ys[i] <= ys[i + 1]:
            local_min.append((xs[i], ys[i]))
        elif ys[i - 1] < ys[i] >= ys[i + 1]:
            local_max.append((xs[i], ys[i]))

    i_min = min(range(n + 1), key=ys.__getitem__)
    i_max = max(range(n + 1), key=ys.__getitem__)
    return {
        "local_min": local_min,
        "local_max": local_max,
        "global_min": (xs[i_min], ys[i_min]),
        "global_max": (xs[i_max], ys[i_max]),
        "interval": (a, b),
    }


res = find_extrema(math.sin, 0, 2 * math.pi, n=1000)
print(res["local_max"], res["local_min"])

[(1.5707963267948966, 1.0)] [(4.71238898038469, -1.0)]


**Тестовые функции:**

| Функция | Отрезок | Что ожидаем |
|---------|---------|-------------|
| $\sin x$ | $[0,\ 4\pi]$ | по 2 локальных минимума и максимума, глобальные — внутри отрезка |
| $x^3 - 3x$ | $[-2,\ 3]$ | локальный максимум в $x=-1$, минимум в $x=1$; глобальный максимум — **на границе** $x = 3$ |
| Функция Растригина $10 + x^2 - 10\cos(2\pi x)$ | $[-5.12,\ 5.12]$ | много локальных минимумов, глобальный минимум в $x = 0$ |

In [24]:
def cubic(x):
    """x^3 - 3x."""
    return x**3 - 3 * x

def rastrigin(x):
    """Одномерная функция Растригина."""
    return 10 + x**2 - 10 * math.cos(2 * math.pi * x)

### 2.1. Настройка логгера (1 балл)

Допишите функцию `setup_logger(name, filename, *, file_level=logging.DEBUG, console_level=logging.WARNING)`, которая возвращает логгер с именем `name` и:
1. уровнем самого логгера `DEBUG` (всё решают обработчики);
2. обработчиком `logging.FileHandler(filename, mode="w", encoding="utf-8")` с уровнем `file_level`;
3. обработчиком `logging.StreamHandler(sys.stdout)` с уровнем `console_level`;
4. общим форматом строки `"%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"` и форматом времени `"%H:%M:%S"`.

Уже сделано в заготовке (разберитесь, зачем):
- `logger.handlers.clear()` — при повторном выполнении ячейки в Colab `getLogger` вернёт **тот же** логгер, и без очистки обработчики добавятся повторно: каждое сообщение будет выводиться дважды, трижды…;
- `logger.propagate = False` — не передавать сообщения корневому логгеру, который в Colab уже настроен (иначе сообщения продублируются).

In [25]:
def setup_logger(name, filename, *, file_level=logging.DEBUG, console_level=logging.WARNING):
    logger = logging.getLogger(name)
    # Close old files, not merely detach them, when rerunning the cell.
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    logger.propagate = False
    logger.setLevel(logging.DEBUG)
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s", datefmt="%H:%M:%S"
    )
    file_handler = logging.FileHandler(filename, mode="w", encoding="utf-8")
    file_handler.setLevel(file_level)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(console_level)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    return logger

In [26]:
# ПРОВЕРКА 2.1 — не изменяйте эту ячейку

log = setup_logger("extrema", "extrema.log")
log = setup_logger("extrema", "extrema.log")          # повторный вызов не должен дублировать обработчики

assert log.level == logging.DEBUG
assert len(log.handlers) == 2, "должно быть ровно два обработчика"
levels = sorted(h.level for h in log.handlers)
assert levels == [logging.DEBUG, logging.WARNING], levels
assert any(isinstance(h, logging.FileHandler) for h in log.handlers)
print("2.1 — все проверки пройдены")

2.1 — все проверки пройдены


### 2.2. Декораторы `log_call` и `warn_boundary` (2 балла)

**Декоратор `log_call`** — универсальный, подходит для **любой** функции. Сигнатура:

```python
def log_call(func=None, *, logger=None, level=logging.INFO): ...
```

Работает в обеих формах: `@log_call` и `@log_call(logger=log, level=logging.INFO)`. Если `logger` не передан, используется `logging.getLogger(func.__module__)`.

При каждом вызове обёртка:

| Когда | Уровень | Сообщение (пример) |
|-------|---------|--------------------|
| до вызова | `DEBUG` | `Вызов find_extrema_logged(sin, 0, 12.566, n=500)` |
| после успешного вызова | `DEBUG` | `find_extrema_logged выполнена за 0.0012 с` |
| после успешного вызова | `level` | `find_extrema_logged вернула {'global_max': (...), ...}` |
| если функция выбросила исключение | `ERROR` | `Ошибка в find_extrema_logged(sin, 5, 1)` + трассировка |

Требования:
- аргументы в сообщении форматируются вспомогательной функцией `short_repr` (дана ниже): для функций выводится имя, для остального — сокращённое представление;
- результат сокращается через `reprlib.repr(result)`, чтобы длинные списки не забивали журнал;
- исключение записывается методом `logger.exception(...)` (он добавляет трассировку и использует уровень `ERROR`), после чего **выбрасывается повторно** (`raise`): декоратор не должен «глотать» ошибки;
- используйте форматирование `logger.debug("... %s", value)`, а не f-строки.

**Декоратор `warn_boundary(logger)`** — предметный, с **обязательным** аргументом (фабрика декораторов, лекция, раздел 7). Он вызывает функцию и проверяет результат: если глобальный минимум или максимум достигается на **конце** отрезка (`result["interval"]`), пишет сообщение уровня `WARNING`, например:

```
find_extrema_logged: global_max на границе отрезка: x=3, f(x)=18 — это может быть не экстремум функции, а край отрезка
```

Для сравнения чисел с плавающей точкой используйте `math.isclose(x, a, abs_tol=1e-12)`.

**Порядок применения:** `@log_call(...)` — внешний декоратор, `@warn_boundary(...)` — внутренний.

In [27]:
def short_repr(value):
    """Имя для функций, сокращённый repr для остальных значений."""
    if callable(value) and hasattr(value, "__name__"):
        return value.__name__
    if isinstance(value, float):
        return f"{value:.5g}"
    return reprlib.repr(value)

print(short_repr(math.sin), short_repr(math.pi), short_repr(list(range(100))))

sin 3.1416 [0, 1, 2, 3, 4, 5, ...]


In [28]:
def log_call(func=None, *, logger=None, level=logging.INFO):
    if func is None:
        return lambda original: log_call(original, logger=logger, level=level)
    chosen_logger = logger if logger is not None else logging.getLogger(func.__module__)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        parts = [short_repr(value) for value in args]
        parts.extend("%s=%s" % (key, short_repr(value)) for key, value in kwargs.items())
        arguments = ", ".join(parts)
        chosen_logger.debug("Вызов %s(%s)", func.__name__, arguments)
        started = time.perf_counter()
        try:
            result = func(*args, **kwargs)
        except Exception:
            chosen_logger.exception("Ошибка в %s(%s)", func.__name__, arguments)
            raise
        elapsed = time.perf_counter() - started
        chosen_logger.debug("%s выполнена за %.6f с", func.__name__, elapsed)
        chosen_logger.log(level, "%s вернула %s", func.__name__, reprlib.repr(result))
        return result
    return wrapper


def warn_boundary(logger):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            a, b = result["interval"]
            for kind in ("global_min", "global_max"):
                x, y = result[kind]
                if (math.isclose(x, a, rel_tol=0.0, abs_tol=1e-12)
                        or math.isclose(x, b, rel_tol=0.0, abs_tol=1e-12)):
                    logger.warning(
                        "%s: %s на границе отрезка: x=%s, f(x)=%s — "
                        "это может быть не экстремум функции, а край отрезка",
                        func.__name__, kind, x, y,
                    )
            return result
        return wrapper
    return decorator


@log_call(logger=log)
@warn_boundary(log)
def find_extrema_logged(f, a, b, *, n=1000):
    """find_extrema с журналированием."""
    return find_extrema(f, a, b, n=n)

In [29]:
# ПРОВЕРКА 2.2 — не изменяйте эту ячейку

class ListHandler(logging.Handler):
    """Обработчик, который складывает записи в список — удобно для тестов."""
    def __init__(self):
        super().__init__(logging.DEBUG)
        self.records = []
    def emit(self, record):
        self.records.append(record)

test_log = logging.getLogger("test_2_2")
test_log.handlers.clear(); test_log.propagate = False; test_log.setLevel(logging.DEBUG)
catcher = ListHandler(); test_log.addHandler(catcher)

@log_call(logger=test_log)
def div(a, b):
    """Деление."""
    return a / b

assert div(6, 3) == 2, "log_call должен возвращать результат"
assert div.__name__ == "div" and div.__doc__ == "Деление."
lv = [r.levelname for r in catcher.records]
assert lv == ["DEBUG", "DEBUG", "INFO"], f"ожидались DEBUG, DEBUG, INFO, получено {lv}"
assert "div(6, 3)" in catcher.records[0].getMessage()

catcher.records.clear()
try:
    div(1, 0)
except ZeroDivisionError:
    pass
else:
    raise AssertionError("log_call не должен «глотать» исключения")
assert catcher.records[-1].levelname == "ERROR" and catcher.records[-1].exc_info, \
    "используйте logger.exception"

@log_call                                   # форма без скобок
def ident(x):
    return x
assert ident(5) == 5

catcher.records.clear()
@warn_boundary(test_log)
def fe(f, a, b, *, n=1000):
    return find_extrema(f, a, b, n=n)

fe(cubic, -2, 3)
warnings_ = [r.getMessage() for r in catcher.records if r.levelname == "WARNING"]
assert any("global_max" in m for m in warnings_), "нет WARNING о глобальном максимуме на границе"
catcher.records.clear()
fe(math.sin, 0, 4 * math.pi)
assert not catcher.records, "для sin на [0, 4π] предупреждений быть не должно"
print("2.2 — все проверки пройдены")

2.2 — все проверки пройдены


### 2.3. Запуск и анализ журнала (1 балл)

1. Вызовите `find_extrema_logged` для трёх тестовых функций из таблицы выше (для Растригина возьмите `n=5000`).
2. Вызовите `find_extrema_logged(math.sin, 5, 1)` — с неверным отрезком. Перехватите исключение в ячейке с помощью `try/except`, чтобы ноутбук выполнялся дальше.
3. Выведите содержимое файла `extrema.log`.
4. Посчитайте, сколько записей каждого уровня в журнале (заготовка ниже использует `collections.Counter`).

Обратите внимание, что на экран (в вывод ячейки) попадают **только** сообщения `WARNING` и `ERROR`, а в файл — все.

**❓ Вопросы:**
1. Почему для $x^3 - 3x$ на $[-2, 3]$ появилось предупреждение и о **минимуме**? Проверьте значения $f(-2)$ и $f(1)$.
2. Что изменится в журнале, если поменять декораторы местами (`@warn_boundary` снаружи, `@log_call` внутри)? Проверьте.

In [30]:
results = {
    "sin": find_extrema_logged(math.sin, 0, 4 * math.pi),
    "cubic": find_extrema_logged(cubic, -2, 3),
    "rastrigin": find_extrema_logged(rastrigin, -5.12, 5.12, n=5000),
}
for name, result in results.items():
    print(name, "local_min:", len(result["local_min"]),
          "local_max:", len(result["local_max"]),
          "global_min:", result["global_min"], "global_max:", result["global_max"])
try:
    find_extrema_logged(math.sin, 5, 1)
except ValueError as error:
    print("Ожидаемая ошибка перехвачена:", error)

from collections import Counter
for handler in log.handlers:
    handler.flush()
with open("extrema.log", encoding="utf-8") as fh:
    text = fh.read()
print(text)
counts = Counter(line.split(" | ")[1].strip() for line in text.splitlines() if " | " in line)
print("Количество записей:", counts)

# Separate logger: preserve the main log while checking decorator order.
order_log = logging.getLogger("order_demo")
order_log.handlers.clear()
order_log.propagate = False
order_log.setLevel(logging.DEBUG)
order_handler = ListHandler()
order_log.addHandler(order_handler)
normal = log_call(logger=order_log)(warn_boundary(order_log)(find_extrema))
reverse = warn_boundary(order_log)(log_call(logger=order_log)(find_extrema))
normal(cubic, -2, 3)
normal_order = [record.levelname for record in order_handler.records]
order_handler.records.clear()
reverse(cubic, -2, 3)
reverse_order = [record.levelname for record in order_handler.records]
print("log_call снаружи:", normal_order)
print("warn_boundary снаружи:", reverse_order)
assert normal_order == ["DEBUG", "WARNING", "WARNING", "DEBUG", "INFO"]
assert reverse_order == ["DEBUG", "DEBUG", "INFO", "WARNING", "WARNING"]
assert cubic(-2) == cubic(1) == -2

22:37:33 | WARNING  | extrema | find_extrema_logged: global_min на границе отрезка: x=-2.0, f(x)=-2.0 — это может быть не экстремум функции, а край отрезка


22:37:33 | WARNING  | extrema | find_extrema_logged: global_max на границе отрезка: x=3.0, f(x)=18.0 — это может быть не экстремум функции, а край отрезка


sin local_min: 2 local_max: 2 global_min: (4.71238898038469, -1.0) global_max: (1.5707963267948966, 1.0)
cubic local_min: 1 local_max: 1 global_min: (-2.0, -2.0) global_max: (3.0, 18.0)
rastrigin local_min: 11 local_max: 10 global_min: (0.0, 0.0) global_max: (-4.521984, 40.3530920250807)
22:37:33 | ERROR    | extrema | Ошибка в find_extrema_logged(sin, 5, 1)
Traceback (most recent call last):
  File "/tmp/ipykernel_6579/1407611770.py", line 14, in wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6579/1407611770.py", line 29, in wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6579/1407611770.py", line 49, in find_extrema_logged
    return find_extrema(f, a, b, n=n)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6579/3284135185.py", line 4, in find_extrema
    raise ValueError(f"левая граница должна быть меньше правой: a={a}, b={b}")
ValueError: левая граница должна быть

Ожидаемая ошибка перехвачена: левая граница должна быть меньше правой: a=5, b=1


22:37:33 | DEBUG    | extrema | Вызов find_extrema_logged(sin, 0, 12.566)
22:37:33 | DEBUG    | extrema | find_extrema_logged выполнена за 0.000261 с
22:37:33 | INFO     | extrema | find_extrema_logged вернула {'global_max': (1.5707963267948966, 1.0), 'global_min': (4.71238898038469, -1.0), 'interval': (0, 12.566370614359172), 'local_max': [(1.5707963267948966, 1.0), (7.853981633974484, 1.0)], ...}
22:37:33 | DEBUG    | extrema | Вызов find_extrema_logged(cubic, -2, 3)
22:37:33 | WARNING  | extrema | find_extrema_logged: global_min на границе отрезка: x=-2.0, f(x)=-2.0 — это может быть не экстремум функции, а край отрезка
22:37:33 | WARNING  | extrema | find_extrema_logged: global_max на границе отрезка: x=3.0, f(x)=18.0 — это может быть не экстремум функции, а край отрезка
22:37:33 | DEBUG    | extrema | find_extrema_logged выполнена за 0.001584 с
22:37:33 | INFO     | extrema | find_extrema_logged вернула {'global_max': (3.0, 18.0), 'global_min': (-2.0, -2.0), 'interval': (-2, 3), 'l

In [31]:
# ПРОВЕРКА 2.3 — не изменяйте эту ячейку

with open("extrema.log", encoding="utf-8") as fh:
    text = fh.read()
for level in ("DEBUG", "INFO", "WARNING", "ERROR"):
    assert f"| {level}" in text, f"в журнале нет записей уровня {level}"
assert "Traceback" in text, "logger.exception должен записать трассировку"
assert "rastrigin" in text and "cubic" in text
print("2.3 — все проверки пройдены")

2.3 — все проверки пройдены


**Ответы на вопросы 2.3:**

1. `f(-2) = -2` и `f(1) = -2`: одинаковый глобальный минимум достигается и на границе,
и внутри. `min` возвращает первый индекс в сетке, поэтому выбран `x=-2` и декоратор
выдал предупреждение. Оно не утверждает, что внутренних минимумов нет.
2. При `log_call` снаружи предупреждения появляются между записью вызова и записями
времени/результата. При обратном порядке они появляются после INFO о результате;
измеренное время внутреннего `log_call` не включает работу внешнего `warn_boundary`.
Оба порядка реально проверены отдельным обработчиком в предыдущей ячейке.

### 2.4. Эксперимент с уровнями (1 балл)

1. Создайте логгер `setup_logger("extrema_info", "extrema_info.log", file_level=logging.INFO)`.
2. Задекорируйте `find_extrema` заново, но уже с этим логгером (через вызовы декораторов, без `@`: `log_call(logger=...)(warn_boundary(...)(find_extrema))`).
3. Вызовите новую функцию для $\sin x$ на $[0, 4\pi]$ и выведите файл `extrema_info.log`.
4. Установите **уровень самого логгера** `WARNING` (`logger.setLevel(logging.WARNING)`), оставив у файлового обработчика уровень `INFO`. Снова вызовите функцию, теперь для $x^3 - 3x$ на $[-2, 3]$, и выведите файл.

**❓ Вопросы:**
1. Каких записей нет в `extrema_info.log` после шага 3 и почему?
2. Почему после шага 4 в файл не попала запись `INFO`, хотя у файлового обработчика уровень `INFO`?
3. Какой уровень вы бы установили для файла и для экрана в «боевой» программе, а какой — при отладке?

In [32]:
info_log = setup_logger("extrema_info", "extrema_info.log", file_level=logging.INFO)
find_extrema_info = log_call(logger=info_log)(warn_boundary(info_log)(find_extrema))
find_extrema_info(math.sin, 0, 4 * math.pi)
for handler in info_log.handlers:
    handler.flush()
with open("extrema_info.log", encoding="utf-8") as fh:
    first_text = fh.read()
print("После sin, logger=DEBUG / file=INFO:\n", first_text)
assert "| DEBUG" not in first_text and first_text.count("| INFO") == 1

info_log.setLevel(logging.WARNING)
find_extrema_info(cubic, -2, 3)
for handler in info_log.handlers:
    handler.flush()
with open("extrema_info.log", encoding="utf-8") as fh:
    final_text = fh.read()
print("После cubic, logger=WARNING / file=INFO:\n", final_text)
assert final_text.count("| INFO") == 1

После sin, logger=DEBUG / file=INFO:
 22:37:33 | INFO     | extrema_info | find_extrema вернула {'global_max': (1.5707963267948966, 1.0), 'global_min': (4.71238898038469, -1.0), 'interval': (0, 12.566370614359172), 'local_max': [(1.5707963267948966, 1.0), (7.853981633974484, 1.0)], ...}

22:37:33 | WARNING  | extrema_info | find_extrema: global_min на границе отрезка: x=-2.0, f(x)=-2.0 — это может быть не экстремум функции, а край отрезка


22:37:33 | WARNING  | extrema_info | find_extrema: global_max на границе отрезка: x=3.0, f(x)=18.0 — это может быть не экстремум функции, а край отрезка


После cubic, logger=WARNING / file=INFO:
 22:37:33 | INFO     | extrema_info | find_extrema вернула {'global_max': (1.5707963267948966, 1.0), 'global_min': (4.71238898038469, -1.0), 'interval': (0, 12.566370614359172), 'local_max': [(1.5707963267948966, 1.0), (7.853981633974484, 1.0)], ...}
22:37:33 | WARNING  | extrema_info | find_extrema: global_min на границе отрезка: x=-2.0, f(x)=-2.0 — это может быть не экстремум функции, а край отрезка
22:37:33 | WARNING  | extrema_info | find_extrema: global_max на границе отрезка: x=3.0, f(x)=18.0 — это может быть не экстремум функции, а край отрезка



In [33]:
# ПРОВЕРКА 2.4 — не изменяйте эту ячейку

with open("extrema_info.log", encoding="utf-8") as fh:
    text = fh.read()
assert "| DEBUG" not in text, "при file_level=INFO записей DEBUG быть не должно"
assert text.count("| INFO") == 1, "INFO должна быть только от первого вызова (до setLevel(WARNING))"
assert "| WARNING" in text
print("2.4 — все проверки пройдены")

2.4 — все проверки пройдены


**Ответы на вопросы 2.4:**

1. После первого вызова нет DEBUG: файловый обработчик с порогом INFO их отсекает.
Для синуса нет WARNING о границах, поэтому записан только результат INFO.
2. После установки уровня логгера WARNING сообщение INFO отсекается самим логгером
ещё до передачи обработчикам; более низкий порог обработчика не возвращает его.
3. В рабочей программе обычно файл INFO, экран WARNING; при диагностике файл DEBUG,
экран DEBUG или INFO. Пороги зависят от задачи, но в журнал нельзя писать секреты.

### Бонус 2. Медленные вызовы (+1 балл, необязательно)

Добавьте в `log_call` необязательный keyword-only параметр `slow_threshold=None` (в секундах). Если он задан и функция работала дольше, запишите сообщение уровня `WARNING`:

```
find_extrema медленная: 0.2140 с > порога 0.1 с
```

Проверьте на функции Растригина с `n=200_000` и порогом `0.05`. Убедитесь, что старые проверки 2.2 по-прежнему проходят (значение по умолчанию `None` не должно менять поведение).

In [34]:
# Бонус 2 (необязательно)

---
## Контрольные вопросы к защите

1. Что такое декоратор? Как запись `@deco` выглядит без синтаксического сахара?
2. Чем отличаются декоратор без аргументов, с аргументами и с необязательными аргументами? Сколько уровней вложенности у каждого?
3. Где хранится исходная функция внутри декоратора? Как её получить снаружи?
4. Что произойдёт, если в обёртке забыть `return`? А если забыть `@functools.wraps`?
5. В каком порядке применяются и выполняются декораторы в цепочке? Приведите пример из упражнения 2.
6. Какие уровни логирования существуют? Как уровень логгера и уровни обработчиков вместе определяют, куда попадёт сообщение?
7. Почему в `logging` пишут `logger.debug("x=%s", x)`, а не `logger.debug(f"x={x}")`?
8. Почему декоратор не должен «глотать» исключения?